# Out-of-fold calibration

Reliability is measured only on outer-fold predictions. Quantile bins shrink automatically for small samples; Platt and isotonic recalibration are fit inside each training fold before being applied to its test block.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path("research/ml")
sys.path.insert(0, str(ROOT))
SNAPSHOT = ROOT / "data" / "snapshot.csv"
import matplotlib.pyplot as plt
from scanner_ml.data import load_dataset
from scanner_ml.evaluate import calibration_metrics, reliability_table
from scanner_ml.models import out_of_fold_predictions
from scanner_ml.validation import walk_forward_splits
if not SNAPSHOT.exists():
    print("Awaiting production snapshot; run npm run research:export")
else:
    data = load_dataset(SNAPSHOT)
    oof, _ = out_of_fold_predictions(data, walk_forward_splits(data))
    fig, ax = plt.subplots(figsize=(6, 5)); ax.plot([0, 1], [0, 1], "--", color="gray")
    for model in ("pred_logistic", "pred_logistic_platt", "pred_hgb", "pred_hgb_isotonic"):
        usable = oof.dropna(subset=[model])
        if len(usable):
            table = reliability_table(usable.y, usable[model]); ax.plot(table.mean_probability, table.observed_rate, marker="o", label=model)
            print(model, calibration_metrics(usable.y, usable[model]))
    ax.set(xlabel="Predicted probability", ylabel="Observed TP1-first rate"); ax.legend(); plt.show()
